# 07 · Test-vs-test interaction — `foe.interaction`

**Hexkit surface:** `pages/interaction_analysis.py` (`InteractionEngine.run_interaction_analysis`).

FOE fits a full-factorial binomial GLM on aggregated cells (`conversions`, `non_conversions` ~
`C(test1) * C(test2) [* …]`) with statsmodels. R fits the same model with
`glm(cbind(conversions, visitors - conversions) ~ test1 * test2, family = binomial)`.
Every term's coefficient, SE, z and p-value is compared, plus the significance flag.

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import re

import numpy as np
import pandas as pd

from foe.interaction.operations import InteractionEngine
from validation_helpers import Checks

checks = Checks()
engine = InteractionEngine()
ALPHA = 0.05


def r_term(raw):
    """statsmodels/patsy name -> R name, e.g. 'C(test1)[T.B]:C(test2)[T.B]' -> 'test1B:test2B'."""
    if raw == "Intercept":
        return "(Intercept)"
    return ":".join(re.sub(r"C\((\w+)\)\[T\.(.+)\]", r"\1\2", part) for part in raw.split(":"))

## Datasets

In [2]:
two_tests = pd.DataFrame({
    "test1": ["A", "A", "B", "B"], "test2": ["A", "B", "A", "B"],
    "visitors":    [25_000, 24_800, 25_100, 24_900],
    "conversions": [   750,    810,    830,    960],
})
three_level = pd.DataFrame({   # test1 has three arms
    "test1": ["A", "A", "B", "B", "C", "C"], "test2": ["A", "B"] * 3,
    "visitors":    [12_000, 11_900, 12_100, 12_050, 11_950, 12_000],
    "conversions": [   360,    410,    420,    400,    350,    420],
})
rng = np.random.default_rng(9)
cells = pd.MultiIndex.from_product([["A", "B"]] * 3, names=["test1", "test2", "test3"]).to_frame(index=False)
cells["visitors"] = rng.integers(8_000, 9_000, len(cells))
cells["conversions"] = rng.binomial(cells.visitors, 0.03 + 0.004 * (cells.test1 == "B") + 0.003 * (cells.test3 == "B"))
datasets = {"2 tests x 2 arms": (two_tests, ["test1", "test2"]),
            "3-arm x 2-arm": (three_level, ["test1", "test2"]),
            "3 tests x 2 arms": (cells, ["test1", "test2", "test3"])}

In [3]:
%%R
fit_r <- function(df, cols) {
  for (c in cols) df[[c]] <- factor(df[[c]], levels = sort(unique(df[[c]])))
  f <- as.formula(paste("cbind(conversions, visitors - conversions) ~", paste(cols, collapse = " * ")))
  co <- coef(summary(glm(f, family = binomial, data = df)))
  data.frame(term = rownames(co), coef = co[, 1], se = co[, 2], z = co[, 3], p = co[, 4])
}

In [4]:
for name, (df, cols) in datasets.items():
    results, warnings = engine.run_interaction_analysis(df, cols, alpha=ALPHA)
    py = pd.DataFrame(results)
    py["r_term"] = py.raw_term.map(r_term)
    %R -i df -i cols -o out out <- fit_r(df, unlist(cols))
    out = out.set_index("term")
    checks.add_equal(f"{name}: same set of terms", sorted(py.r_term), sorted(out.index))
    for _, row in py.iterrows():
        r = out.loc[row.r_term]
        tag = f"{name}: {row.term}"
        checks.add(f"coef  {tag}", row.coefficient, r.coef, atol=1e-8, rtol=1e-6)
        checks.add(f"SE    {tag}", row.std_err, r.se, rtol=1e-6)
        checks.add(f"z     {tag}", row.z_score, r.z, atol=1e-6, rtol=1e-6)
        checks.add(f"p     {tag}", row.p_value, r.p, atol=1e-10, rtol=1e-5)
        checks.add_equal(f"sig   {tag}", bool(row.is_significant), bool(r.p < ALPHA))
    display(py[["term", "coefficient", "std_err", "p_value", "is_significant"]])

,term,coefficient,std_err,p_value,is_significant
0,Baseline (Control Group),-3.476099,0.037075,0.000000,True
1,test1 (B),0.100528,0.051192,0.049558,True
2,test2 (B),0.087741,0.051486,0.088351,False
3,test1 (B) & test2 (B) — Clash/Synergy,0.071457,0.070571,0.311272,False


,term,coefficient,std_err,p_value,is_significant
0,Baseline (Control Group),-3.476099,0.053513,0.000000,True
1,test1 (B),0.150720,0.073009,0.038978,True
2,test1 (C),-0.024729,0.076204,0.745556,False
3,test2 (B),0.143023,0.073415,0.051397,False
4,test1 (B) & test2 (B) — Clash/Synergy,-0.189242,0.102187,0.064037,False
5,test1 (C) & test2 (B) — Clash/Synergy,0.041024,0.103925,0.693032,False


,term,coefficient,std_err,p_value,is_significant
0,Baseline (Control Group),-3.528931,0.065490,0.000000,True
1,test1 (B),0.152159,0.090265,0.091855,False
2,test2 (B),-0.051153,0.092298,0.579430,False
3,test3 (B),0.223936,0.087123,0.010160,True
4,test1 (B) & test2 (B) — Clash/Synergy,0.035632,0.126661,0.778464,False
5,test1 (B) & test3 (B) — Clash/Synergy,-0.140472,0.121723,0.248490,False
6,test2 (B) & test3 (B) — Clash/Synergy,0.098427,0.123301,0.424713,False
7,test1 (B) & test2 (B) & test3 (B) — Clash/Synergy,0.055596,0.170290,0.744062,False


## Summary

In [5]:
checks.report()

93 checks, 93 passed, 0 failed

Largest differences:
                                                        check   hexkit        R    abs_diff              tolerance
             z     2 tests x 2 arms: Baseline (Control Group) -93.7581 -93.7581  8.6354e-06 atol=1e-06, rtol=1e-06
                            z     2 tests x 2 arms: test1 (B)  1.96376  1.96376 1.63721e-07 atol=1e-06, rtol=1e-06
                            z     2 tests x 2 arms: test2 (B)  1.70416  1.70416 1.45082e-07 atol=1e-06, rtol=1e-06
z     2 tests x 2 arms: test1 (B) & test2 (B) — Clash/Synergy  1.01256  1.01256 7.62783e-08 atol=1e-06, rtol=1e-06
p     2 tests x 2 arms: test1 (B) & test2 (B) — Clash/Synergy 0.311272 0.311271 3.64507e-08 atol=1e-10, rtol=1e-05


,check,hexkit,R,abs_diff,tolerance,pass
0,2 tests x 2 arms: same set of terms,"[(Intercept), test1B, test1B:test2B, test2B]","[(Intercept), test1B, test1B:test2B, test2B]",NaN,exact,True
1,coef 2 tests x 2 arms: Baseline (Control Group),-3.476099,-3.476099,2.176037e-14,"atol=1e-08, rtol=1e-06",True
2,SE 2 tests x 2 arms: Baseline (Control Group),0.037075,0.037075,3.414737e-09,"atol=1e-08, rtol=1e-06",True
3,z 2 tests x 2 arms: Baseline (Control Group),-93.758057,-93.758065,8.635399e-06,"atol=1e-06, rtol=1e-06",True
4,p 2 tests x 2 arms: Baseline (Control Group),0.0,0.0,0.000000e+00,"atol=1e-10, rtol=1e-05",True
...,...,...,...,...,...,...
88,coef 3 tests x 2 arms: test1 (B) & test2 (B) ...,0.055596,0.055596,1.408595e-15,"atol=1e-08, rtol=1e-06",True
89,SE 3 tests x 2 arms: test1 (B) & test2 (B) ...,0.17029,0.17029,2.400191e-12,"atol=1e-08, rtol=1e-06",True
90,z 3 tests x 2 arms: test1 (B) & test2 (B) ...,0.326479,0.326479,4.609924e-12,"atol=1e-06, rtol=1e-06",True
91,p 3 tests x 2 arms: test1 (B) & test2 (B) ...,0.744062,0.744062,3.487322e-12,"atol=1e-10, rtol=1e-05",True
